# Calculator Agentic Workflow

This notebook implements a function-based agentic workflow that can perform basic arithmetic operations:
- Addition
- Subtraction
- Multiplication
- Division

The agent can parse natural language requests and execute the appropriate calculator function.

## 1. Define Calculator Tools

First, we'll define the basic calculator functions that serve as our tools.

In [ ]:
def add(a: float, b: float) -> float:
    """
    Add two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Sum of a and b
    """
    return a + b


def subtract(a: float, b: float) -> float:
    """
    Subtract second number from first number.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Difference of a and b (a - b)
    """
    return a - b


def multiply(a: float, b: float) -> float:
    """
    Multiply two numbers together.
    
    Args:
        a: First number
        b: Second number
    
    Returns:
        Product of a and b
    """
    return a * b


def divide(a: float, b: float) -> float:
    """
    Divide first number by second number.
    
    Args:
        a: First number (dividend)
        b: Second number (divisor)
    
    Returns:
        Quotient of a and b (a / b)
    
    Raises:
        ValueError: If b is zero
    """
    if b == 0:
        raise ValueError("Cannot divide by zero")
    return a / b


print("Calculator tools defined successfully!")

## 2. Define Tool Registry

Create a registry that maps operation names to their corresponding functions.

In [ ]:
CALCULATOR_TOOLS = {
    'add': {
        'function': add,
        'description': 'Add two numbers together',
        'parameters': ['a', 'b'],
        'aliases': ['addition', 'plus', 'sum', '+']
    },
    'subtract': {
        'function': subtract,
        'description': 'Subtract second number from first number',
        'parameters': ['a', 'b'],
        'aliases': ['subtraction', 'minus', 'difference', '-']
    },
    'multiply': {
        'function': multiply,
        'description': 'Multiply two numbers together',
        'parameters': ['a', 'b'],
        'aliases': ['multiplication', 'times', 'product', '*', 'x']
    },
    'divide': {
        'function': divide,
        'description': 'Divide first number by second number',
        'parameters': ['a', 'b'],
        'aliases': ['division', 'divided by', 'quotient', '/']
    }
}

print("Tool registry created!")
print("\nAvailable operations:")
for tool_name, tool_info in CALCULATOR_TOOLS.items():
    print(f"  - {tool_name}: {tool_info['description']}")

## 3. Implement the Calculator Agent

The agent is responsible for:
1. Parsing user requests
2. Identifying the appropriate tool to use
3. Executing the tool with the provided parameters
4. Returning the result

In [ ]:
import re
from typing import Dict, Any, Optional


class CalculatorAgent:
    """
    An agent that can execute calculator operations based on user requests.
    """
    
    def __init__(self, tools: Dict[str, Dict[str, Any]]):
        """
        Initialize the calculator agent with available tools.
        
        Args:
            tools: Dictionary of available calculator tools
        """
        self.tools = tools
        self.history = []
    
    def _identify_operation(self, request: str) -> Optional[str]:
        """
        Identify which operation the user wants to perform.
        
        Args:
            request: User's request string
        
        Returns:
            The operation name, or None if not found
        """
        request_lower = request.lower()
        
        # Check each tool and its aliases
        for tool_name, tool_info in self.tools.items():
            if tool_name in request_lower:
                return tool_name
            for alias in tool_info['aliases']:
                if alias in request_lower:
                    return tool_name
        
        return None
    
    def _extract_numbers(self, request: str) -> list:
        """
        Extract numbers from the request string.
        
        Args:
            request: User's request string
        
        Returns:
            List of numbers found in the request
        """
        # Find all numbers (including decimals and negatives)
        numbers = re.findall(r'-?\d+\.?\d*', request)
        return [float(n) for n in numbers if n]
    
    def execute(self, operation: str, a: float, b: float) -> Dict[str, Any]:
        """
        Execute a calculator operation.
        
        Args:
            operation: Name of the operation to perform
            a: First operand
            b: Second operand
        
        Returns:
            Dictionary containing the result and execution details
        """
        if operation not in self.tools:
            return {
                'success': False,
                'error': f"Unknown operation: {operation}",
                'available_operations': list(self.tools.keys())
            }
        
        tool = self.tools[operation]
        
        try:
            result = tool['function'](a, b)
            
            execution_record = {
                'success': True,
                'operation': operation,
                'inputs': {'a': a, 'b': b},
                'result': result
            }
            
            # Add to history
            self.history.append(execution_record)
            
            return execution_record
        
        except Exception as e:
            return {
                'success': False,
                'operation': operation,
                'inputs': {'a': a, 'b': b},
                'error': str(e)
            }
    
    def process_request(self, request: str) -> Dict[str, Any]:
        """
        Process a natural language request.
        
        Args:
            request: User's natural language request
        
        Returns:
            Dictionary containing the result and execution details
        """
        # Identify the operation
        operation = self._identify_operation(request)
        
        if not operation:
            return {
                'success': False,
                'error': 'Could not identify operation from request',
                'request': request,
                'available_operations': list(self.tools.keys())
            }
        
        # Extract numbers
        numbers = self._extract_numbers(request)
        
        if len(numbers) < 2:
            return {
                'success': False,
                'error': 'Could not extract two numbers from request',
                'request': request,
                'numbers_found': numbers
            }
        
        # Execute the operation with the first two numbers
        return self.execute(operation, numbers[0], numbers[1])
    
    def get_history(self) -> list:
        """
        Get the execution history.
        
        Returns:
            List of execution records
        """
        return self.history
    
    def clear_history(self):
        """
        Clear the execution history.
        """
        self.history = []


print("Calculator Agent class defined successfully!")

## 4. Initialize the Agent

Create an instance of the calculator agent.

In [ ]:
# Create the agent
agent = CalculatorAgent(CALCULATOR_TOOLS)

print("Calculator Agent initialized and ready to use!")

## 5. Test the Agent

Let's test the agent with various requests.

In [ ]:
# Test with structured execution
print("=== Structured Execution Tests ===")
print()

# Addition
result = agent.execute('add', 15, 7)
print(f"15 + 7 = {result['result']}")
print(f"Full result: {result}")
print()

# Subtraction
result = agent.execute('subtract', 20, 8)
print(f"20 - 8 = {result['result']}")
print(f"Full result: {result}")
print()

# Multiplication
result = agent.execute('multiply', 6, 9)
print(f"6 * 9 = {result['result']}")
print(f"Full result: {result}")
print()

# Division
result = agent.execute('divide', 100, 4)
print(f"100 / 4 = {result['result']}")
print(f"Full result: {result}")
print()

# Division by zero (error handling)
result = agent.execute('divide', 10, 0)
print(f"10 / 0: {result}")
print()

In [ ]:
# Test with natural language requests
print("=== Natural Language Request Tests ===")
print()

test_requests = [
    "Add 25 and 17",
    "What is 50 minus 23?",
    "Multiply 12 and 8",
    "Divide 144 by 12",
    "What is 3.14 plus 2.86?",
    "Calculate the product of 7 and 9",
    "What's the difference between 100 and 37?",
    "What is 15.5 times 2?"
]

for request in test_requests:
    result = agent.process_request(request)
    if result['success']:
        print(f"Request: {request}")
        print(f"Result: {result['inputs']['a']} {result['operation']} {result['inputs']['b']} = {result['result']}")
    else:
        print(f"Request: {request}")
        print(f"Error: {result['error']}")
    print()

## 6. View Execution History

The agent keeps track of all successful operations.

In [ ]:
print("=== Execution History ===")
print()

history = agent.get_history()
print(f"Total operations executed: {len(history)}")
print()

for i, record in enumerate(history, 1):
    print(f"{i}. {record['operation'].upper()}: {record['inputs']['a']} and {record['inputs']['b']} = {record['result']}")

print()

## 7. Interactive Mode

Use this cell to interactively test the calculator agent with your own requests.

In [ ]:
# Interactive calculator
def interactive_calculator():
    """
    Run an interactive calculator session.
    """
    print("Interactive Calculator Agent")
    print("=" * 50)
    print("Enter your calculation requests (or 'quit' to exit)")
    print("Examples:")
    print("  - Add 5 and 3")
    print("  - What is 10 minus 4?")
    print("  - Multiply 6 by 7")
    print("  - Divide 20 by 5")
    print("=" * 50)
    print()
    
    while True:
        request = input("Enter request: ").strip()
        
        if request.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not request:
            continue
        
        result = agent.process_request(request)
        
        if result['success']:
            print(f"Result: {result['result']}")
        else:
            print(f"Error: {result['error']}")
        
        print()

# Uncomment the line below to run interactive mode
# interactive_calculator()

## 8. Advanced Usage: Chained Operations

You can chain multiple operations together.

In [ ]:
print("=== Chained Operations ===")
print()

# Example: Calculate (10 + 5) * 3 / 5
print("Calculating: (10 + 5) * 3 / 5")
print()

step1 = agent.execute('add', 10, 5)
print(f"Step 1: 10 + 5 = {step1['result']}")

step2 = agent.execute('multiply', step1['result'], 3)
print(f"Step 2: {step1['result']} * 3 = {step2['result']}")

step3 = agent.execute('divide', step2['result'], 5)
print(f"Step 3: {step2['result']} / 5 = {step3['result']}")

print()
print(f"Final result: {step3['result']}")

## Summary

This notebook demonstrates a function-based agentic workflow for a calculator that:

1. **Defines calculator tools**: Four basic arithmetic operations (add, subtract, multiply, divide)
2. **Implements an agent**: The `CalculatorAgent` class that can:
   - Parse natural language requests
   - Identify the appropriate operation
   - Extract numbers from text
   - Execute operations
   - Track execution history
   - Handle errors gracefully
3. **Supports multiple interfaces**:
   - Structured execution: `agent.execute(operation, a, b)`
   - Natural language: `agent.process_request("Add 5 and 3")`
   - Interactive mode: For real-time calculations

The agent-based approach makes the calculator flexible and extensible, allowing for easy addition of new operations and features.